# AI Academic Advisor — DATA308 Generative AI, Assignment #1

**"Design, Build, Deploy and Evaluate an AI Academic Advisor for University that knows when to
answer, what to answer, when to ask and when to say it does not have enough information."**

This notebook is the experimental workbench for Phases 1–5. Every cell is one step.

| Phase | What it does | Cells | Marks |
|---|---|---|---|
| 1 | Build the RAG advisor | 2–9 | 10 |
| 2 | Challenge it with synthetic profiles & edge cases | 10–12 | 5 |
| 3 | Improve through prompt engineering | 13–14 | 5 |
| 4 | Evaluate and measure (34 verified test cases) | 15–19 | 10 |
| 5 | Compare and analyse across the four variants | 20–24 | 5 |
| 6 | Deploy (`app.py`, see final cell) | 25 | 5 |
| 7 | Report & PPT (see `report/`) | — | 5 |

**Architecture in one line:** numbers are *computed* by a deterministic rule engine,
prose is *retrieved* from the documents — the LLM only explains and cites, it never calculates.

## Cell 1 — Setup

Install dependencies. On Colab, also upload the four source documents.

In [ ]:
# If running on Google Colab, uncomment the next line:
# !pip -q install google-genai pdfplumber openpyxl scikit-learn streamlit

import sys, os, warnings, textwrap
from pathlib import Path
warnings.filterwarnings("ignore")

# Point this at the project folder (the one containing the `advisor/` package).
PROJECT = Path.cwd()
if not (PROJECT / "advisor").exists():
    PROJECT = Path("/content/ai-academic-advisor")   # typical Colab path
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print("project root:", PROJECT)
print("source docs :", sorted(p.name for p in (PROJECT/"data"/"raw").glob("*")))

## Cell 2 — API key

Get a **free** Gemini key at <https://aistudio.google.com/apikey>.

The whole pipeline degrades gracefully without a key: retrieval falls back to TF-IDF and
generation is disabled, so you can still inspect ingestion, the rule engine and the test set.

In [ ]:
import getpass, os
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google Gemini API key: ").strip()

from advisor.llm import LLMClient
_llm = LLMClient()
print("key detected:", _llm.available, "| model:", _llm.model)
if _llm.available:
    print("ping ->", _llm.generate("Be terse.", "Reply with exactly: READY").text)

## Cell 3 — Phase 1a · Ingest the four university documents

The ingestion layer splits the corpus deliberately in two:

* **Structured** (`courses.csv`, `baskets.csv`, `minors.csv`) — course codes, credits,
  prerequisites, semester offerings, basket minimums. Facts an LLM must never paraphrase.
* **Unstructured** (`chunks.jsonl`) — regulation prose, chunked so that a chunk never
  straddles a top-level clause boundary and always carries `source + clause + page`.
  Without that metadata, citations cannot be checked, and "source correctness" is
  one of the eight graded metrics.

In [ ]:
from advisor.ingest import build_all
artefacts = build_all()

## Cell 4 — Phase 1b · Inspect the parsed catalogue

Always look at parsed data before trusting it.

In [ ]:
import pandas as pd
courses, baskets, minors = artefacts["courses"], artefacts["baskets"], artefacts["minors"]

print("Courses per batch per semester:")
display(pd.crosstab(courses.batch, courses.semester))

print("\nCredit requirements per basket:")
display(baskets.pivot_table(index="basket", columns="batch", values="min_credits"))

print("\nA sample of courses that carry prerequisites:")
display(courses[courses.prereq_raw.str.upper() != "NIL"]
        [["batch","semester","course_code","title","prereq_raw","credits"]].head(10))

## Cell 5 — Phase 1c · Build the hybrid retriever

Academic queries mix two very different retrieval needs:

* exact-token lookups — `DATA301`, `Clause 7.2`, `65%` → **BM25** wins
* paraphrased concepts — *"can I retake a subject I failed?"* → **dense embeddings** win

So scores from both are min-max normalised and blended. Crucially, the *abstention*
decision does **not** use the blended score: min-max normalisation always forces the top
hit to 1.0, which would make "no relevant document" impossible to detect. Abstention uses
**absolute** signals instead — raw cosine similarity plus query-term coverage.

In [ ]:
from advisor.retriever import HybridRetriever
retriever = HybridRetriever()
print(f"index mode = {retriever.mode} | {len(retriever.chunks)} passages\n")

for q in ["What is the minimum attendance requirement?",
          "prerequisites for Machine Learning DATA301",
          "What CGPA do I need for Year 3?",
          "How much does a canteen meal cost?"]:      # <- deliberately out of corpus
    hits = retriever.search(q, k=3)
    print(f"Q: {q}")
    print(f"   -> out_of_corpus = {retriever.is_out_of_corpus(hits)}")
    for h in hits[:2]:
        print(f"      {h.score:.2f} cos={h.dense:.2f} cov={h.coverage:.2f}  {h.chunk.citation()}")
    print()

## Cell 6 — Phase 1d · The deterministic rule engine

This is the design decision that matters most. LLMs are unreliable at multi-step
arithmetic and at rule application, and both are exactly what academic advising needs.
So eligibility, credits, attendance and progression are **computed in Python** from the
structured catalogue and the student record, and the *result* is handed to the model as a
verified fact it is forbidden to recompute.

Every rule cites the handbook clause it came from.

In [ ]:
from advisor.eligibility import EligibilityEngine, StudentRecord
engine = EligibilityEngine()

demo = StudentRecord("DEMO", "2025", "B.Tech CSE (Data Science)", 4, cgpa=6.1,
                     completed={"DATA103":"A", "MATH209":"A", "MATH301":"F", "COMP201":"B"},
                     attendance={"MATH301": 58.0, "COMP201": 71.0})

for v in [engine.check_prerequisites(demo, "DATA301"),
          engine.check_attendance(demo),
          engine.check_progression(demo),
          engine.failed_course_actions(demo)]:
    print(f"### {v.question}"); print(v.to_prompt_block()); print()

## Cell 7 — Phase 1e · Automatic conflict detection

Phase 2 asks the advisor to *identify conflicts*. Rather than inventing them, we mine the
real ones out of the university's own spreadsheet. Three kinds are detected:

* `DANGLING_PREREQ` — a course requires a course its own batch never offers
* `ORDERING` — a prerequisite is offered no earlier than the course itself
* `CROSS_BATCH` — the same course code has different prerequisites per batch

These are genuine defects in the published curriculum, and they are the strongest
"conflicting rules" test material available.

In [ ]:
from advisor import conflicts
found = conflicts.detect()
print(f"{len(found)} conflicts detected in the published curriculum\n")
for kind in ("CROSS_BATCH", "ORDERING", "DANGLING_PREREQ"):
    sub = [c for c in found if c.kind == kind]
    print(f"--- {kind} ({len(sub)}) ---")
    for c in sub[:3]:
        print(textwrap.fill(c.line(), 110, subsequent_indent="      ")); print()
conflicts.save_report()

## Cell 8 — Phase 1f · First end-to-end answer

Variant **V4** = retrieval + verified student record + conflict detection + follow-up protocol.
Note what happens on an under-specified question: the advisor *asks* rather than assuming.

In [ ]:
from advisor.advisor import AcademicAdvisor
from advisor.eligibility import load_students
from advisor.synthetic import main as generate_profiles

if not (PROJECT/"data"/"processed"/"synthetic_students.json").exists():
    generate_profiles()
students = {s.student_id: s for s in load_students()}

advisor_v4 = AcademicAdvisor("V4_rag_student", retriever=retriever, engine=engine)

def ask(question, student=None, adv=None):
    a = (adv or advisor_v4).answer(question, student=student)
    print("Q:", question)
    print("-"*100)
    print(a.text if a.ok else f"[call failed] {a.error}")
    print("-"*100)
    print(f"latency {a.latency_s:.2f}s | abstained={a.abstained} | "
          f"follow-up={a.asked_followup} | citations={len(a.citations)}\n")
    return a

_ = ask("Can I take Machine Learning next semester?")          # under-specified -> should ASK

## Cell 9 — Phase 1g · The same question, now with a student record

With the record supplied, the advisor should stop asking and start deciding.

In [ ]:
_ = ask("Can I take Analysis of Algorithms next semester?", student=students["STU-A"])
print("Rule-engine facts that were injected into the prompt:\n")
print(engine.profile_brief(students["STU-A"], mentioned=["COMP203"])[:1400])

## Cell 10 — Phase 2 · Synthetic student profiles

Ten anonymised profiles. No names, no IDs, no contact details — opaque tokens only.
Course codes and credits are real, so the eligibility arithmetic is genuine.

Each profile targets one edge case the assignment names: missing information, ambiguity,
insufficient prerequisites, insufficient credits, unavailable courses, conflicting rules.

In [ ]:
import json
profiles = generate_profiles()

## Cell 11 — Phase 2b · Edge-case coverage matrix

Which profile exercises which failure mode.

In [ ]:
rows = []
for p in profiles:
    cs = engine.credit_summary(p)
    rows.append({
        "id": p.student_id, "batch": p.batch, "sem": f"S{p.current_semester}",
        "CGPA": p.cgpa, "passed": len(p.passed()), "failed": len(p.failed()),
        "credits": cs.facts["earned"],
        "progression": engine.check_progression(p).decision,
        "attendance": engine.check_attendance(p).decision,
        "edge case": p.notes.replace("Edge case: ", ""),
    })
display(pd.DataFrame(rows).set_index("id"))

## Cell 12 — Phase 2c · Run the hard scenarios

Four deliberately difficult questions across four different profiles.

In [ ]:
scenarios = [
    ("STU-A", "I failed Data Structures. Can I register for Analysis of Algorithms next semester?"),
    ("STU-B", "I have a clean record. Am I eligible for DATA301 Machine Learning?"),
    ("STU-C", "My CGPA is 3.84 after Semester 2. Will I be promoted to Year 2?"),
    ("STU-G", "Will I be promoted to the next year?"),   # CGPA missing -> must ask
]
for sid, q in scenarios:
    print(f"########## {sid} ##########")
    _ = ask(q, student=students[sid])

## Cell 13 — Phase 3 · Prompt engineering, one ingredient at a time

Four variants, each adding exactly **one** thing to the previous one, so any metric change
is attributable to that ingredient:

| Variant | Adds |
|---|---|
| V1 Basic LLM | nothing — plain question to the model |
| V2 Structured Prompting | role, context, constraints, delimiters, output contract |
| V3 RAG | retrieved evidence + citation duty + abstention duty |
| V4 RAG + Structured Student Data | verified rule-engine facts + conflicts + follow-up protocol |

In [ ]:
from advisor import prompts
from advisor.advisor import build_all_variants

variants = build_all_variants(api_key=os.getenv("GOOGLE_API_KEY"))
for name, sysmsg in prompts.SYSTEMS.items():
    print(f"{prompts.VARIANT_LABELS[name]:<40} system prompt = {len(sysmsg):>5} chars")

## Cell 14 — Phase 3b · Same question through all four variants

A question the corpus cannot answer. Watch V1/V2 invent a number and V3/V4 refuse.

In [ ]:
probe = "How many grade points does an 'A+' carry on the 10-point scale?"
for name, adv in variants.items():
    a = adv.answer(probe, student=students["STU-A"])
    print(f"===== {prompts.VARIANT_LABELS[name]} =====")
    print(textwrap.fill((a.text or a.error)[:700], 108))
    print(f"[abstained={a.abstained}  latency={a.latency_s:.2f}s]\n")

## Cell 15 — Phase 4 · The test set

34 cases, each with a **verified** expected outcome traced to a clause. Two scoring signals
carry most of the weight:

* `must_include` — facts a correct answer has to contain
* `must_not_include` — **hallucination traps**: plausible but *wrong* values. If one appears,
  the answer is scored incorrect (or unsupported, where the case expected an abstention).
  These traps are what make the hallucination rate measurable rather than impressionistic.

In [ ]:
from advisor import testset
cases = testset.load(); testset.save()
s = testset.summary()
print(f"{s['total']} cases\n")
display(pd.Series(s["by_category"], name="cases").to_frame())
display(pd.Series(s["by_behaviour"], name="cases").to_frame())

print("\nExample — a hallucination trap:")
c = [t for t in cases if t.id == "M01"][0]
print(f"  Q            : {c.question}")
print(f"  behaviour    : {c.behaviour}")
print(f"  expected     : {c.expected_answer}")
print(f"  traps        : {c.must_not_include}")

## Cell 16 — Phase 4b · Scoring methodology

Each response lands in exactly one bucket, by deterministic rule:

| Label | Meaning |
|---|---|
| `correct` | right action **and** every required fact present |
| `partially_correct` | right action, some required facts missing |
| `unsupported` | asserted content where the corpus has none — a hallucination |
| `incorrect` | contradicted a verified fact, or took the wrong action |

**Correctness** (does it match the verified rule) is reported as *accuracy*.
**Reliability** (does it stay grounded, admit gaps, avoid unsupported recommendations)
is reported as *hallucination rate*, *missing-info handling* and *conflict handling*.
A system can be correct on easy questions and unreliable on ambiguous ones, so both are shown.

In [ ]:
from advisor import evaluate
import inspect
print(inspect.getsource(evaluate.score_case))

## Cell 17 — Phase 4c · Run the full evaluation

⏱ 34 cases × 4 variants ≈ 136 model calls. Expect a few minutes.
`sleep` throttles requests to stay inside the Gemini free-tier rate limit.

In [ ]:
all_results = {}
for name, adv in variants.items():
    print(f"\n===== {prompts.VARIANT_LABELS[name]} =====")
    all_results[name] = evaluate.run_variant(adv, cases, students, sleep=1.2)

paths = evaluate.save_results(all_results)
print("\nsaved:", {k: str(v.name) for k, v in paths.items()})

## Cell 18 — Phase 4d · The eight metrics

This table is the core evidence for Phases 4 and 5. Put it in the report.

In [ ]:
table = evaluate.comparison_table(all_results)
pd.set_option("display.width", 220)
display(table.T)

## Cell 19 — Phase 4e · Outcome mix per variant

In [ ]:
from dataclasses import asdict
raw = pd.DataFrame([asdict(r) for res in all_results.values() for r in res])
raw["variant"] = raw.variant.map(prompts.VARIANT_LABELS).fillna(raw.variant)
mix = (raw.pivot_table(index="variant", columns="label", values="case_id",
                       aggfunc="count").fillna(0).astype(int))
display(mix)

## Cell 20 — Phase 5 · Charts

Basic LLM → Structured Prompting → RAG → RAG + Structured Data.

In [ ]:
import matplotlib.pyplot as plt
order = [prompts.VARIANT_LABELS[v] for v in prompts.VARIANTS]
t = table.reindex(order)
short = ["V1 Basic", "V2 Prompted", "V3 RAG", "V4 RAG+Data"]

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
panels = [
    ("accuracy_%",            "Accuracy (%)",                   "#4a6fa5"),
    ("hallucination_rate_%",  "Hallucination rate (%) — lower is better", "#c0504d"),
    ("missing_info_handled_%","Missing information handled (%)", "#548235"),
    ("source_correct_%",      "Source / evidence correctness (%)", "#8064a2"),
]
for a, (col, title, colour) in zip(ax.ravel(), panels):
    vals = t[col].astype(float).values
    bars = a.bar(short, vals, color=colour)
    a.set_title(title, fontsize=11); a.set_ylim(0, 105); a.grid(axis="y", alpha=.3)
    a.bar_label(bars, fmt="%.0f", padding=2, fontsize=9)
plt.tight_layout(); plt.savefig("results/phase5_progression.png", dpi=160)
plt.show()

## Cell 21 — Phase 5b · Where each variant wins and loses, by category

In [ ]:
cat = evaluate.per_category(all_results)[order]
display(cat)

fig, ax = plt.subplots(figsize=(11, 5))
cat.plot(kind="bar", ax=ax, width=.8, colormap="viridis")
ax.set_ylabel("% correct"); ax.set_ylim(0, 105)
ax.set_title("Correctness by question category"); ax.grid(axis="y", alpha=.3)
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.savefig("results/phase5_by_category.png", dpi=160); plt.show()

## Cell 22 — Phase 5c · Handling of missing information

The assignment requires **at least 3** questions the provided data cannot answer.
We have four (M01–M04), plus two out-of-scope cases (S01–S02).

`M01` is the most interesting: the letter-grade → grade-point table (Table 2, Clause 8.10)
is an **image** in the handbook PDF, so its contents never reach the text layer. A system
that answers "A+ = 9 points" is reciting general knowledge about Indian universities, not
this university's regulation. That is precisely the failure mode the assignment targets.

In [ ]:
miss = raw[raw.category.isin(["missing_information", "out_of_scope"])]
display(miss.pivot_table(index="case_id", columns="variant",
                         values="label", aggfunc="first"))

print("\nVerbatim responses to M01 (grade points — genuinely absent from the corpus):\n")
for r in raw[raw.case_id == "M01"].itertuples():
    print(f"--- {r.variant} -> {r.label} ---")
    print(textwrap.fill(str(r.answer)[:420], 108)); print()

## Cell 23 — Phase 5d · Error analysis: what still fails

Every non-`correct` case for the best variant. This drives the Limitations section of the report.

In [ ]:
best = prompts.VARIANT_LABELS["V4_rag_student"]
bad = raw[(raw.variant == best) & (raw.label != "correct")]
print(f"{len(bad)} of {len(cases)} cases still not fully correct for {best}\n")
for r in bad.itertuples():
    print(f"[{r.case_id}] {r.category} -> {r.label}")
    print(f"   expected {r.behaviour_expected}, observed {r.behaviour_observed}; "
          f"matched {r.matched}/{r.required}"
          + (f"; TRAP HIT {r.traps_hit}" if r.traps_hit else ""))
    print(f"   Q: {r.question}")
    print(textwrap.fill(f"   A: {str(r.answer)[:260]}", 108)); print()

## Cell 24 — Phase 5e · Retrieval ablation

Does the *hybrid* actually help, or would BM25 alone do? Sweep the fusion weight and measure
how often the correct source document is retrieved in the top-k.
`w=1.0` is pure BM25, `w=0.0` is pure dense.

In [ ]:
from advisor import config as cfg
probes = [(t.question, t.expected_source) for t in cases
          if not t.expected_source.lower().startswith("no source")]

def hit_rate(w, k=6):
    old, cfg.BM25_WEIGHT = cfg.BM25_WEIGHT, w
    ok = 0
    for q, src in probes:
        blob = " ".join(h.chunk.citation() for h in retriever.search(q, k=k))
        key = ("Programme Structure" if "Programme" in src or "Semester" in src
               else "SOP" if "SOP" in src
               else "Minor" if "Minor" in src else "Student Handbook")
        ok += key.lower() in blob.lower()
    cfg.BM25_WEIGHT = old
    return round(100 * ok / len(probes), 1)

abl = pd.DataFrame({"BM25 weight": [0.0, 0.25, 0.5, 0.75, 1.0]})
abl["source doc in top-6 (%)"] = [hit_rate(w) for w in abl["BM25 weight"]]
display(abl)
abl.plot(x="BM25 weight", y="source doc in top-6 (%)", marker="o", figsize=(7,3.5),
         title="Retrieval ablation: lexical vs dense", grid=True, legend=False)
plt.tight_layout(); plt.savefig("results/retrieval_ablation.png", dpi=160); plt.show()

## Cell 25 — Phase 6 · Deploy

```bash
streamlit run app.py                       # local
```

**Streamlit Community Cloud (free public demo link for the report):**

1. Push this folder to a **public** GitHub repo.
2. Go to <https://share.streamlit.io> → *New app* → pick the repo → main file `app.py`.
3. *Advanced settings → Secrets*, paste:
   ```toml
   GOOGLE_API_KEY = "your-key-here"
   ```
4. Deploy. Put the resulting URL on the cover page of the report.

> Never commit the key. `.gitignore` already excludes `.streamlit/secrets.toml`.

In [ ]:
print("Artefacts produced for the report:\n")
for p in sorted(Path("results").glob("*")):
    print(f"  results/{p.name}")
for p in sorted(Path("data/processed").glob("*.csv")) + sorted(Path("data/processed").glob("*.json")):
    print(f"  {p}")